✍ В стандартной библиотеке Python для отправки веб-запросов существует функция urllib2, но большинство разработчиков используют стороннюю библиотеку requests (c англ. запросы), потому что её работа более стабильна, а созданный с её помощью код получается проще. Поэтому мы будем работать с библиотекой requests, а urllib2 рассматривать не будем.

Познакомимся с библиотекой requests, решив простую задачу — получить значения курсов валют. Курс валют — полезная и регулярно обновляемая информация, но каждый раз в ручном режиме получать информацию о курсе интересующей валюты трудоёмко.

Разработаем код, так называемый скрипт (англ. script, рус. сценарий), — небольшую программу, которая содержит последовательность действий для автоматического выполнения задачи.

С помощью скрипта мы будем в удобном виде выгружать информацию по курсам валют с заранее выбранного сайта.

In [2]:
# Установка библиотеки requests
%pip install requests

Note: you may need to restart the kernel to use updated packages.


Как только библиотека установлена, импортируем её и отправим наш первый запрос к ресурсу **Курсы валют ЦБ РФ в XML и JSON**. Используем метод get() из библиотеки requests, передав ему соответствующий URL —  https://www.cbr-xml-daily.ru/daily_json.js:

In [8]:
import requests
url = 'https://www.cbr-xml-daily.ru/daily_json.js'
response = requests.get(url)

In [9]:
print(response)

<Response [200]>


Мы получили объект ответа Response, который содержит всю нужную нам информацию. По умолчанию в квадратных скобках на экран выводится код статуса ответа. В данном случае он равен 200 — то есть запрос был корректным и сервер отдал нам нужную информацию. Значение кода статуса 404 означало бы, что страница по указанному адресу не найдена, а значение 403 — что синтаксис GET-запроса неверный.

Код ответа в виде числовой переменной можно получить с помощью метода status_code:

In [10]:
print(response.status_code)

200


In [6]:
response = requests.get(url='https://www.cbr-xml-daily.ru/daily.xml')
print(response.status_code)

200


РАБОТАЕМ С ОТВЕТОМ

Мы сделали запрос и получили корректный ответ (код статуса — 200). Дальнейшую работу производим с результатом запроса к ресурсу Курсы валют ЦБ РФ в XML и JSON.

Как получить доступ ко всей информации, которую содержит ответ?

Текст ответа хранится в атрибуте text. Выведем значение атрибута на экран и посмотрим на его содержимое:

In [11]:
print(response.text)

{"Date":"2026-02-28T11:30:00+03:00","PreviousDate":"2026-02-27T11:30:00+03:00","PreviousURL":"\/\/www.cbr-xml-daily.ru\/archive\/2026\/02\/27\/daily_json.js","Timestamp":"2026-02-28T11:00:00+03:00","Valute":{"AUD":{"ID":"R01010","NumCode":"036","CharCode":"AUD","Nominal":1,"Name":"Австралийский доллар","Value":55.0652,"Previous":54.9647},"AZN":{"ID":"R01020A","NumCode":"944","CharCode":"AZN","Nominal":1,"Name":"Азербайджанский манат","Value":45.4551,"Previous":45.3658},"DZD":{"ID":"R01030","NumCode":"012","CharCode":"DZD","Nominal":100,"Name":"Алжирских динаров","Value":59.5434,"Previous":59.4237},"GBP":{"ID":"R01035","NumCode":"826","CharCode":"GBP","Nominal":1,"Name":"Фунт стерлингов","Value":104.4353,"Previous":104.3766},"AMD":{"ID":"R01060","NumCode":"051","CharCode":"AMD","Nominal":100,"Name":"Армянских драмов","Value":20.497,"Previous":20.4545},"BHD":{"ID":"R01080","NumCode":"048","CharCode":"BHD","Nominal":1,"Name":"Бахрейнский динар","Value":205.4705,"Previous":205.0669},"BYN":

Как правило, при работе над реальным проектом на этапе получения данных мы уже понимаем, с какими форматами данных нам придётся работать. На предлагаемом для работы ресурсе информация есть как в JSON-формате, так и в XML. По нашему запросу ресурс возвращает информацию в JSON-формате, однако в настоящий момент результат хранится как единая строка. Проверить тип данных полученного ответа можно, воспользовавшись функцией type().

Для того чтобы удобно было работать с полученной информацией, нам необходимо преобразовать строку в словарь. В объект ответа Response  из библиотеки requests уже встроен метод json() .

Импортируем функцию pprint(), применим к полученному ответу метод json() и выведем полученный результат на экран:

In [12]:
from pprint import pprint
import json

currencies = response.json()
pprint(currencies)

{'Date': '2026-02-28T11:30:00+03:00',
 'PreviousDate': '2026-02-27T11:30:00+03:00',
 'PreviousURL': '//www.cbr-xml-daily.ru/archive/2026/02/27/daily_json.js',
 'Timestamp': '2026-02-28T11:00:00+03:00',
 'Valute': {'AED': {'CharCode': 'AED',
                    'ID': 'R01230',
                    'Name': 'Дирхам ОАЭ',
                    'Nominal': 1,
                    'NumCode': '784',
                    'Previous': 20.9998,
                    'Value': 21.0411},
            'AMD': {'CharCode': 'AMD',
                    'ID': 'R01060',
                    'Name': 'Армянских драмов',
                    'Nominal': 100,
                    'NumCode': '051',
                    'Previous': 20.4545,
                    'Value': 20.497},
            'AUD': {'CharCode': 'AUD',
                    'ID': 'R01010',
                    'Name': 'Австралийский доллар',
                    'Nominal': 1,
                    'NumCode': '036',
                    'Previous': 54.9647,
             

Теперь данные находятся в словаре и можно легко получать необходимые значения.

Например, по ключу Valute мы можем обратиться к вложенному словарю, который содержит информацию о мировых валютах. Выведем на экран, например, информацию о евро (EUR):

In [14]:
pprint(currencies['Valute']['EUR'])

{'CharCode': 'EUR',
 'ID': 'R01239',
 'Name': 'Евро',
 'Nominal': 1,
 'NumCode': '978',
 'Previous': 91.0281,
 'Value': 91.2965}


In [15]:
pprint(currencies['Valute']['CZK']['Name'])

'Чешских крон'
